# AHC015 128-channel DDP microbatch benchmark

Kaggle T4 x2上で、蒸留後の未来列なし128 channel checkpointを使い、蒸留なしPPO更新のglobal microbatch `256 / 512 / 1024`を比較する。effective batch sizeはすべて1024。256 episodesの同一rolloutに対して各条件を2回測定し、本学習は行わない。所要時間の目安は数分。GPU T4 x2、Internet On、`GITHUB_TOKEN`と`WANDB_API_KEY`のSecret accessを有効にしてSave & Run Allする。

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA GPU is required"
assert torch.cuda.device_count() == 2, "Select the Kaggle GPU T4 x2 accelerator"
for index in range(2):
    name = torch.cuda.get_device_name(index)
    capability = torch.cuda.get_device_capability(index)
    print(index, name, capability)
    assert "T4" in name and capability == (7, 5), "GPU T4 x2 is required"
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
assert not repo_dir.exists(), f"Clean session required: {repo_dir} already exists"
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"
try:
    subprocess.run(
        ["git", "clone", "--branch", "feature/ahc015-teacher",
         "--single-branch", "https://github.com/e1jirou/ahc-ml.git", str(repo_dir)],
        check=True, env=git_env,
    )
finally:
    del github_token, credentials, git_env
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
os.chdir(repo_dir)
print("Repository commit:", actual_commit)

In [ ]:
import wandb

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
del wandb_api_key
assert wandb.login(verify=True)
api = wandb.Api()
source_run = api.run("eijirou-personal/ahc-ml/v1fy4rlz")
assert source_run.name == "distill-20260903-045216"
assert source_run.state == "finished"
artifact = api.artifact(
    "eijirou-personal/ahc-ml/distill-20260903-045216-training-checkpoint:v0",
    type="model",
)
checkpoint_dir = Path("/kaggle/working/checkpoints") / source_run.name
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
checkpoint_path = downloaded_dir / "best-training.pt"
assert checkpoint_path.is_file()
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
assert checkpoint["epoch"] == 23
assert checkpoint["config"]["model"]["channels"] == 128
assert checkpoint["config"]["model"]["future_mode"] == "none"
print("Checkpoint:", checkpoint_path)
del checkpoint

In [ ]:
import sys

benchmark_env = os.environ.copy()
benchmark_env["PYTHONPATH"] = str(repo_dir / "python")
subprocess.run(
    [
        sys.executable, "-m",
        "examples.ahc015.python.benchmark_afterstate_microbatch",
        "--checkpoint", str(checkpoint_path),
        "--episodes", "256",
        "--repeats", "2",
        "--micro-batch-sizes", "256", "512", "1024",
    ],
    cwd=repo_dir, env=benchmark_env, check=True,
)
print("Benchmark completed; no training run was started.")